# LeetCode #1136: Parallel Courses

https://leetcode.com/problems/parallel-courses/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n! \cdot (n + m))$ | $O(n)$ |
| **Optimal: Topological Sort (Kahn's BFS) ★** | $O(n + m)$ | $O(n + m)$ |

---

## Understanding the Methods

### Brute Force
Try every ordering of courses, check prerequisites, and track the semester count. Factorial explosion makes this impossible for $n > 10$.

### Optimal: Topological Sort (Kahn's BFS) ★
Use Kahn's algorithm: enqueue all courses with no prerequisites (in-degree 0). Each BFS level corresponds to one semester — take all currently unlocked courses simultaneously. If a cycle exists, not all courses are taken and we return -1.

**Constraints:**
* $1 \le n \le 10^4$
* $0 \le relations.length \le 5 \times 10^4$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    public int MinimumSemesters(int n, int[][] relations) {
        var adj = new List<int>[n + 1];
        for (int i = 1; i <= n; i++) adj[i] = new List<int>();
        var indegree = new int[n + 1];

        foreach (var r in relations) {
            adj[r[0]].Add(r[1]);
            indegree[r[1]]++;
        }

        // Start with all courses that have no prerequisites
        var q = new Queue<int>();
        for (int i = 1; i <= n; i++)
            if (indegree[i] == 0) q.Enqueue(i);

        int semesters = 0, taken = 0;
        while (q.Count > 0) {
            // All courses in the queue can be taken this semester
            int size = q.Count;
            semesters++;
            for (int i = 0; i < size; i++) {
                int course = q.Dequeue();
                taken++;
                foreach (int next in adj[course]) {
                    if (--indegree[next] == 0) q.Enqueue(next); // prereqs satisfied
                }
            }
        }
        // If we couldn't take all courses, there's a cycle
        return taken == n ? semesters : -1;
    }
}

### Python

In [ ]:
from collections import deque

class Solution:
    def minimumSemesters(self, n: int, relations: list[list[int]]) -> int:
        adj = [[] for _ in range(n + 1)]
        indegree = [0] * (n + 1)

        for pre, next_course in relations:
            adj[pre].append(next_course)
            indegree[next_course] += 1

        # Enqueue all courses with no prerequisites
        q = deque(i for i in range(1, n + 1) if indegree[i] == 0)
        semesters = 0
        taken = 0

        while q:
            # Every course currently in the queue is taken this semester
            for _ in range(len(q)):
                course = q.popleft()
                taken += 1
                for nxt in adj[course]:
                    indegree[nxt] -= 1
                    if indegree[nxt] == 0:
                        q.append(nxt)  # all prerequisites now satisfied
            semesters += 1

        return semesters if taken == n else -1  # -1 signals a cycle

### Go

In [ ]:
func minimumSemesters(n int, relations [][]int) int {
	adj := make([][]int, n+1)
	indegree := make([]int, n+1)
	for _, r := range relations {
		adj[r[0]] = append(adj[r[0]], r[1])
		indegree[r[1]]++
	}

	// All courses with no prerequisites can start immediately
	q := []int{}
	for i := 1; i <= n; i++ {
		if indegree[i] == 0 {
			q = append(q, i)
		}
	}

	semesters, taken := 0, 0
	for len(q) > 0 {
		semesters++
		next := []int{}
		for _, course := range q {
			taken++
			for _, nxt := range adj[course] {
				indegree[nxt]--
				if indegree[nxt] == 0 {
					next = append(next, nxt) // prerequisite satisfied
				}
			}
		}
		q = next
	}
	if taken == n {
		return semesters
	}
	return -1 // cycle detected
}

### Rust

In [ ]:
use std::collections::VecDeque;

impl Solution {
    pub fn minimum_semesters(n: i32, relations: Vec<Vec<i32>>) -> i32 {
        let n = n as usize;
        let mut adj = vec![vec![]; n + 1];
        let mut indegree = vec![0i32; n + 1];

        for r in &relations {
            adj[r[0] as usize].push(r[1] as usize);
            indegree[r[1] as usize] += 1;
        }

        // All courses with in-degree 0 are immediately available
        let mut q: VecDeque<usize> = (1..=n).filter(|&i| indegree[i] == 0).collect();
        let mut semesters = 0;
        let mut taken = 0;

        while !q.is_empty() {
            semesters += 1;
            let level_size = q.len();
            for _ in 0..level_size {
                let course = q.pop_front().unwrap();
                taken += 1;
                for &nxt in &adj[course] {
                    indegree[nxt] -= 1;
                    if indegree[nxt] == 0 {
                        q.push_back(nxt); // all prerequisites satisfied
                    }
                }
            }
        }
        if taken == n { semesters } else { -1 }
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=3, relations=[[1,3],[2,3]]`
Courses 1 and 2 have no prerequisites → semester 1. Course 3 unlocked → semester 2. Answer: **2**.

### 2. Slightly Complex
**Input:** `n=3, relations=[[1,2],[2,3]]`
Strict chain: semester 1 → course 1, semester 2 → course 2, semester 3 → course 3. Answer: **3**.

### 3. Edge Case: Time Factor
**Input:** `n=10000, relations` forms a balanced binary DAG of depth $\log_2 n$.
Kahn's BFS processes all $n + m$ edges in $O(n + m)$, far faster than any exhaustive search.

### 4. Edge Case: Space Factor
**Input:** `n=10000, relations` is 50 000 random prerequisite pairs.
Adjacency list uses $O(n + m)$ space; the in-degree array is $O(n)$; the BFS queue holds at most $O(n)$ nodes simultaneously.

### 5. Almost-Impossible but Plausible
**Input:** `n=3, relations=[[1,2],[2,3],[3,1]]` (cycle)
No course reaches in-degree 0. Queue is empty from the start, `taken=0 \ne n=3`. Returns **-1** — correctly detecting the cycle in $O(n + m)$.